# 인터넷전문은행 중·저신용자 대출 규제 실효성 분석

분석 질문 4개를 검증하기 위한 코드 골격입니다. 섹션별로 순서대로 실행하세요.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (환경에 맞게 경로/폰트명 조정 필요)
plt.rcParams['axes.unicode_minus'] = False
# plt.rcParams['font.family'] = 'AppleGothic'  # macOS
# plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows


## 1. 데이터 로드

In [ ]:
DATA_PATH = "../data/데이터북_v4.xlsx"  # 실제 파일명으로 교체

df_credit = pd.read_excel(DATA_PATH, sheet_name="월별_평균신용점수", skiprows=3)
df_rate = pd.read_excel(DATA_PATH, sheet_name="월별_고금리비중", skiprows=3)

df_credit.head()


In [ ]:
df_rate.head()


## 2. 데이터 기본 정보 확인\n- 기간, 컬럼, 결측치 여부 확인

In [ ]:
for name, df in [("월별_평균신용점수", df_credit), ("월별_고금리비중", df_rate)]:
    print(f"=== {name} ===")
    print("행 수:", len(df))
    print("컬럼:", list(df.columns))
    print("결측치:\n", df.isna().sum())
    print()


In [ ]:
# 취급월을 datetime으로 변환
df_credit['취급월'] = pd.to_datetime(df_credit['취급월'])
df_rate['취급월'] = pd.to_datetime(df_rate['취급월'])

print("신용점수 데이터 기간:", df_credit['취급월'].min(), "~", df_credit['취급월'].max())
print("고금리비중 데이터 기간:", df_rate['취급월'].min(), "~", df_rate['취급월'].max())


## 3. 결측치 / 이상치 처리\n- 처리 기준을 정하고 근거를 마크다운으로 함께 남길 것\n- 2024년 지표 정의 변경 등 구조적 단절 여부 확인

In [ ]:
# TODO: 결측치/이상치 처리
# TODO: 정의 변경 시점(구조적 단절) 마킹


## 4. 시계열 분석 기법 적용 (2개 이상)

In [ ]:
# 4-1. 이동평균 (예: 3개월)
df_credit_wide = df_credit.pivot(index='취급월', columns='은행', values='평균신용점수')
df_credit_ma = df_credit_wide.rolling(window=3).mean()
df_credit_ma.head()


In [ ]:
# 4-2. 전월 대비 변화율
df_credit_pct = df_credit_wide.pct_change() * 100
df_credit_pct.head()


## 5. 시각화

In [ ]:
# 시각화 1: 질문 1 - 3사 평균신용점수 추이
fig, ax = plt.subplots(figsize=(10, 5))
for bank in df_credit_wide.columns:
    ax.plot(df_credit_wide.index, df_credit_wide[bank], marker='o', label=bank)
ax.set_title('3사 월별 평균신용점수 추이')
ax.set_xlabel('취급월')
ax.set_ylabel('평균신용점수')
ax.legend()
plt.tight_layout()
plt.savefig('../images/01_credit_score_trend.png', dpi=150)
plt.show()


In [ ]:
# 시각화 2: 질문 2 - 3사 고금리 비중 추이
df_rate_wide = df_rate.pivot(index='취급월', columns='은행', values='고금리8퍼센트이상비중')

fig, ax = plt.subplots(figsize=(10, 5))
for bank in df_rate_wide.columns:
    ax.plot(df_rate_wide.index, df_rate_wide[bank], marker='o', label=bank)
ax.set_title('3사 월별 고금리(8%↑) 취급비중 추이')
ax.set_xlabel('취급월')
ax.set_ylabel('고금리 비중(%)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/02_high_rate_share_trend.png', dpi=150)
plt.show()


In [ ]:
# 시각화 3 (권장): 질문 3 - 신용점수 변화율 vs 고금리비중 변화율 비교
# TODO: 두 지표를 정규화해서 같은 축에 겹쳐 그리기


In [ ]:
# 시각화 4 (선택): 질문 4 - 서민금융제외평균금리-평균금리 스프레드
df_credit['스프레드'] = df_credit['서민금융제외평균금리'] - df_credit['평균금리']
df_spread_wide = df_credit.pivot(index='취급월', columns='은행', values='스프레드')

fig, ax = plt.subplots(figsize=(10, 5))
for bank in df_spread_wide.columns:
    ax.plot(df_spread_wide.index, df_spread_wide[bank], marker='o', label=bank)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_title('서민금융제외평균금리 - 평균금리 스프레드')
ax.set_xlabel('취급월')
ax.set_ylabel('스프레드(%p)')
ax.legend()
plt.tight_layout()
plt.savefig('../images/03_spread_trend.png', dpi=150)
plt.show()


## 6. 인사이트 도출\n- 관찰(Fact) / 원인(Why, 가설) / 행동(Action) 구조로 정리 → REPORT.md에 옮겨 작성

In [ ]:
# TODO: 인사이트 근거가 될 구체적 수치 계산
